In [ ]:
# @title 🤖 Installations and GPU Checks

# Make sure runtime is set to using T4 (should be free and available if you have not used them all)
import torch
if torch.cuda.is_available():
    print(f"✅ GPU is active: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU not found. Check your Runtime settings!")

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate sentence-transformers faiss-cpu gradio huggingface_hub rank_bm25

In [ ]:
from huggingface_hub import notebook_login, hf_hub_download, login
from google.colab import userdata
import os

# You must add your own Hugging Face token to the secrets tab called HF_TOKEN.
# The Llama and Gemma models require access, so be sure to request access on Hugging Face first before trying them.

token = userdata.get('HF_TOKEN')
login(token=token)

In [ ]:
# @title 🤖 Model Selection Configuration
MODEL_CHOICE = "Gemma-2 (9B)" # @param ["Llama-3 (8B)", "Gemma-2 (9B)", "Mistral-Mini (7B)"]

# Central repository for FAISS, BM25, and text chunks
DATA_REPO = "Hsweazey/bot-llama"

model_configs = {
    "Llama-3 (8B)": {
        "base": "meta-llama/Meta-Llama-3-8B-Instruct",
        "adapter": "Hsweazey/bot-llama"
    },
    "Gemma-2 (9B)": {
        "base": "google/gemma-2-9b-it",
        "adapter": "Hsweazey/bot-gemma"
    },
    "Mistral-Mini (7B)": {
        "base": "mistralai/Mistral-7B-Instruct-v0.3",
        "adapter": "Hsweazey/bot-mistral"
    }
}

selected = model_configs[MODEL_CHOICE]
print(f"✅ Ready to load {MODEL_CHOICE} with data from {DATA_REPO}")

In [ ]:
import torch
import gc

def clear_gpu_memory():
    global model, base_model, tokenizer
    if 'model' in globals():
        del model
    if 'base_model' in globals():
        del base_model
    if 'tokenizer' in globals():
        del tokenizer

    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU Memory Cleared. You can now load a new model.")

# Run this if you are switching models
clear_gpu_memory()

In [ ]:
# @title 🤖 Download Data from Hugging Face Space
import pickle
import json
import faiss

# Download the files
for filename in ["faiss.index", "chunks.json", "bm25.pkl"]:
    hf_hub_download(repo_id=DATA_REPO, filename=f"{filename}", local_dir=".", token=token)

# Load them into memory
print("--- Loading Retrieval Assets ---")
with open("chunks.json", "r") as f:
    text_chunks = json.load(f)

with open("bm25.pkl", "rb") as f:
    bm25_obj = pickle.load(f)

index = faiss.read_index("faiss.index")

In [ ]:
# @title 🤖 Retrieval Logic
import numpy as np

from sentence_transformers import SentenceTransformer, CrossEncoder

print("--- Initializing Retrieval Engines ---")

# 1. The Embedding Model (for FAISS/Semantic Search)
# This must match the model used to create your index on Sciclone
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. The Cross-Encoder (for Reranking/The 'Judge')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("✅ Retrieval Engines Ready.")
import numpy as np

def hybrid_retrieve(query, top_k=8):
    # 🔹 1. BM25 (Keyword Search)
    # Tokenize the query and get the top 20 relevant chunks based on exact words
    tokenized_query = query.lower().split()
    bm25_scores = bm25_obj.get_scores(tokenized_query)
    bm25_top = np.argsort(bm25_scores)[::-1][:20]

    # 🔹 2. FAISS (Semantic Search)
    # Convert query to an embedding and find top 20 based on "meaning"
    query_emb = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(query_emb)
    distances, faiss_top = index.search(query_emb, 20)
    faiss_top = faiss_top[0]

    # 🔹 3. Combine & Deduplicate
    # We take the union of both lists and sort them for determinism
    candidates = sorted(list(set(bm25_top).union(set(faiss_top))))

    # 🔹 4. Cross-Encoder Reranking (The "Judge")
    # FIXED: Safe dictionary extraction checking both 'text' and 'content'
    pairs = [[query, text_chunks[i].get('text', text_chunks[i].get('content', ''))] for i in candidates]
    ce_scores = cross_encoder.predict(pairs)

    # Sort candidates by their new, smarter Cross-Encoder scores
    final_idx_positions = np.argsort(ce_scores)[::-1][:top_k]

    results = []
    for pos in final_idx_positions:
        idx = candidates[pos]
        results.append({
            "score": float(ce_scores[pos]),
            # FIXED: Safe extraction for the final context string sent to the LLM
            "text": text_chunks[idx].get('text', text_chunks[idx].get('content', '')),
            "metadata": text_chunks[idx]
        })

    return results

In [ ]:
# @title 🤖 Download Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"--- Downloading & Loading {MODEL_CHOICE} ---")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(selected["base"])
# Ensure padding is consistent across all three architectures
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    selected["base"],
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, selected["adapter"])
print(f"🚀 {MODEL_CHOICE} is now live on the GPU!")

In [ ]:
# @title 💬 W&M Advisor Local Launch

SYSTEM_PROMPT = (
    "You are a professional William & Mary Academic Advisor. Your sole purpose is to assist "
    "students with W&M-related inquiries. If a student asks a question that is unrelated to "
    "William & Mary, you must politely decline to answer and offer to help them with their "
    "academic journey instead. Do not provide general knowledge or instructions outside of this scope."
)

def chat_function(message, history):
    # 1. Retrieve facts using your hybrid engine
    # (Ensure your hybrid_retrieve function is defined in a previous cell)
    retrieved_results = hybrid_retrieve(message)

    # 2. Prepare the context string
    # We use a newline separator to keep the chunks distinct for the model
    retrieved_context_str = "\n\n".join([r['text'] for r in retrieved_results])

    # 3. Standardized RAG Instruction (including the department referral logic)
    user_instruction = f"""Using the following W&M context, answer the student's question. If the context does not contain the answer, politely state that you do not have that information in your current records and suggest the appropriate W&M department, office, or faculty member the student should contact for help. If the question is entirely unrelated to W&M, politely decline to answer.

    Context:
    {retrieved_context_str}

    Student Question:
    {message}"""

    # 4. Handle System Role (Gemma 2 Compatibility Fix)
    if "Gemma" in MODEL_CHOICE:
        # Gemma 2 template does not support the 'system' role; fold it into the user prompt
        messages = [
            {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_instruction}"}
        ]
    else:
        # Llama 3 and Mistral/Mini use the standard system/user split
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_instruction}
        ]

    # 5. Apply the model's specific chat template
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 6. Generate the Advisor's response
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.6,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # 7. Decode and clean up
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Debug: Print the first few words of context to the Colab console
    print(f"🔍 Context used: {retrieved_context_str[:75]}...")

    return response.strip()

# Launch Gradio interface
import gradio as gr
demo = gr.ChatInterface(fn=chat_function, title=f"W&M Advisor Bot ({MODEL_CHOICE})")
demo.launch(share=True, debug=True)